In [11]:
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

True

In [12]:
from langchain_openai import ChatOpenAI
llm_chat_model = ChatOpenAI(model_name="gpt-3.5-turbo-0125")

In [13]:
from langchain_openai import OpenAI
llm_completion_model = OpenAI()

In [14]:
from langchain_core.prompts import PromptTemplate    
prompt_template = PromptTemplate.from_template(
    "Tell me a {adjective} joke about {topic}."
)

llmModelPrompt = prompt_template.format(adjective="poor", topic="modi")
llm_chat_response = llm_chat_model.invoke(llmModelPrompt)
print(llm_chat_response.content)

Why did Modi always carry a ladder with him? 

Because he wanted to climb up the ranks of politics!


# Few shot 

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate
from langchain_core.prompts import ChatPromptTemplate

examples=[
    {"input": "Tell me a joke about a cat.", "output": "Why did the cat sit on the computer? Because it wanted to keep an eye on the mouse!"},
    {"input": "Tell me a joke about a dog.", "output": "Why don't dogs like to go to the beach? Because they already have a tail!"}
]


example_prompt= ChatPromptTemplate.from_messages([
    ("user", "{input}"),
    ("assistant", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    examples=examples,
    example_prompt=example_prompt
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that tells jokes."),
    few_shot_prompt,
    ("user", "{input}")
])

chain = final_prompt | llm_chat_model
chain_response = chain.invoke({"input": "Tell me a joke about a bird."})
print(chain_response.content)


Why do birds fly south in the winter? Because it's too far to walk!


In [26]:
from langchain.output_parsers.json import SimpleJsonOutputParser
json_prompt = PromptTemplate.from_template(
    "Return a JSON object with an 'answer' key that answers the question: {question}"
)
json_output_parser = SimpleJsonOutputParser()
json_chain = json_prompt | llm_chat_model | json_output_parser

json_response = json_chain.invoke({"question": "What is the capital of France?"})
print(json_response)


{'answer': 'The capital of France is Paris.'}


In [27]:
from pydantic import BaseModel
class Answer(BaseModel):
    ans: str



In [31]:
from langchain.output_parsers.pydantic import PydanticOutputParser
parser = PydanticOutputParser(pydantic_object=Answer)
prompt = PromptTemplate.from_template(
    """
    Answer the question.

    {format_instructions}

    Question:
    {question}
    """
)
prompt = prompt.partial(
    format_instructions=parser.get_format_instructions()
)

chain = prompt | llm_chat_model | parser
response = chain.invoke({"question": "What is the capital of France?"})
print(response.model_dump())
print(type(response.model_dump()))

{'ans': 'Paris'}
<class 'dict'>
